In [0]:
%pip install \
    azure-keyvault-secrets==4.7.0 \
    azure-identity==1.15.0 \
    azure-core==1.29.5 \
    azure-storage-file-datalake==12.14.0 \
    sseclient-py \
    openai \
    dotenv \
    confluent-kafka \
    great-expectations \
    altair==4.2.2 \
    redis

In [0]:
import os
import sys
import json
import pandas as pd
from datetime import datetime
from pyspark.sql import functions as F

os.environ["KEY_VAULT_URL"] = "https://kv-sense-team4.vault.azure.net/"
sys.path.insert(0, "/Workspace/Repos/3dt030@msacademy.msai.kr/3dt-3nd-project/src")

import utils.vault_manager
utils.vault_manager._instance = None
from utils.vault_manager import get_vault_manager

vault = get_vault_manager()
vault.get_storage_client("datacopsadls")
print("[OK] 환경설정 완료")

In [0]:
# ADLS에서 CSV 읽기
TEST_CSV_PATH   = "abfss://test@datacopsadls.dfs.core.windows.net/test.csv"
BRONZE_PATH     = "abfss://bronze@datacopsadls.dfs.core.windows.net/dataschool_traffic_accident"

pdf = spark.read \
    .option("header", "true") \
    .option("encoding", "cp949") \
    .csv(TEST_CSV_PATH) \
    .toPandas()

print(f"행 수: {len(pdf)}, 컬럼: {list(pdf.columns)}")

# Bronze 형태로 변환
rows = []
for _, row in pdf.iterrows():
    event = row.where(pd.notna(row), None).to_dict()
    event["_ingest_ts"]   = datetime.utcnow().isoformat() + "Z"
    event["_source_type"] = "batch"
    event["_platform"]    = "datasentinel"
    rows.append((json.dumps(event, ensure_ascii=False),))

# Bronze Delta 적재
df_bronze = spark.createDataFrame(rows, ["raw_json"]) \
    .withColumn("kafka_timestamp",   F.current_timestamp()) \
    .withColumn("_bronze_loaded_at", F.current_timestamp()) \
    .withColumn("_kafka_topic",      F.lit("dataschool.traffic_accident.batch.raw")) \
    .withColumn("_source",           F.col("raw_json"))

df_bronze.write.format("delta").mode("append").save(BRONZE_PATH)
print(f"✅ Bronze 적재 완료: {BRONZE_PATH}")
display(spark.read.format("delta").load(BRONZE_PATH))

In [0]:
# 02 노트북 domains에 추가해서 바로 처리
dbutils.notebook.run(
    "/Workspace/Repos/3dt030@msacademy.msai.kr/3dt-3nd-project/notebooks/02_stream_bronze2silver",
    timeout_seconds=600,
    arguments={"bronze_path": BRONZE_PATH}
)